# Scraping: récupération de la page d'accueil de la RTBF

Dans ce notebook, nous créons un robot qui va ouvrir la page d'accueil du site du journal [La RTBF](https://www.rtbf.be/) et récupérer le titre de tous les articles du jour et les stocker dans un fichier csv.

## Imports

In [91]:
import os
import re
import time
import requests
from bs4 import BeautifulSoup
import pandas as pd

## Récupération de tous les articles de la page d'accueil



In [92]:
headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_10_1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/39.0.2171.95 Safari/537.36'}

pdfs = []

CAMILLE_URL= "https://max.de.wilde.web.ulb.be/camille/"
response = requests.get(CAMILLE_URL, headers=headers)

soup = BeautifulSoup(response.text, 'html.parser')

for link in soup.find_all("a"):
    filename = link.get("href")

    if filename and filename.endswith(".pdf"):
        url = CAMILLE_URL + filename
        pdfs.append([url, filename])

In [93]:
# Affichage du nombre de documents récupérés
len(pdfs)

51

In [94]:
# Affichage des 10 premières entrées
pdfs[:10]

[['https://max.de.wilde.web.ulb.be/camille/KB_JB230_1892-08-07_01-0003.pdf',
  'KB_JB230_1892-08-07_01-0003.pdf'],
 ['https://max.de.wilde.web.ulb.be/camille/KB_JB427_1920-01-10_01-00004.pdf',
  'KB_JB427_1920-01-10_01-00004.pdf'],
 ['https://max.de.wilde.web.ulb.be/camille/KB_JB555_1836-02-08_01-00002.pdf',
  'KB_JB555_1836-02-08_01-00002.pdf'],
 ['https://max.de.wilde.web.ulb.be/camille/KB_JB638_1860-05-21_01-00002.pdf',
  'KB_JB638_1860-05-21_01-00002.pdf'],
 ['https://max.de.wilde.web.ulb.be/camille/KB_JB773_1918-11-30_01-00002.pdf',
  'KB_JB773_1918-11-30_01-00002.pdf'],
 ['https://max.de.wilde.web.ulb.be/camille/KB_JB838_1887-12-28_01-00003.pdf',
  'KB_JB838_1887-12-28_01-00003.pdf'],
 ['https://max.de.wilde.web.ulb.be/camille/KB_JB230_1903-10-16_01-0002.pdf',
  'KB_JB230_1903-10-16_01-0002.pdf'],
 ['https://max.de.wilde.web.ulb.be/camille/KB_JB427_1933-01-04_01-00003.pdf',
  'KB_JB427_1933-01-04_01-00003.pdf'],
 ['https://max.de.wilde.web.ulb.be/camille/KB_JB555_1899-01-19_01-00

## Création d'un dataframe avec les liens et les noms des documents pdf


In [95]:
# La liste pdfs contient déjà, pour chaque document, le lien complet et le nom du fichier
# Il suffit donc de créer le DataFrame avec des noms de colonnes explicites

df = pd.DataFrame(pdfs, columns=["link", "pdf_name"])
df[:10]  # Affichage des 10 premières entrées du DataFrame

,link,pdf_name
0,https://max.de.wilde.web.ulb.be/camille/KB_JB2...,KB_JB230_1892-08-07_01-0003.pdf
1,https://max.de.wilde.web.ulb.be/camille/KB_JB4...,KB_JB427_1920-01-10_01-00004.pdf
2,https://max.de.wilde.web.ulb.be/camille/KB_JB5...,KB_JB555_1836-02-08_01-00002.pdf
3,https://max.de.wilde.web.ulb.be/camille/KB_JB6...,KB_JB638_1860-05-21_01-00002.pdf
4,https://max.de.wilde.web.ulb.be/camille/KB_JB7...,KB_JB773_1918-11-30_01-00002.pdf
5,https://max.de.wilde.web.ulb.be/camille/KB_JB8...,KB_JB838_1887-12-28_01-00003.pdf
6,https://max.de.wilde.web.ulb.be/camille/KB_JB2...,KB_JB230_1903-10-16_01-0002.pdf
7,https://max.de.wilde.web.ulb.be/camille/KB_JB4...,KB_JB427_1933-01-04_01-00003.pdf
8,https://max.de.wilde.web.ulb.be/camille/KB_JB5...,KB_JB555_1899-01-19_01-00003.pdf
9,https://max.de.wilde.web.ulb.be/camille/KB_JB6...,KB_JB638_1902-12-20_01-00002.pdf


In [96]:
# Sauvegarde du dataframe dans un fichier CSV
output_directory = "../data/tp1"
os.makedirs(output_directory, exist_ok=True)

df.to_csv(
    f"{output_directory}/camille_pdfs_{time.strftime('%Y%m%d')}.csv",
    index=False
)

## Téléchargement d'un article et affichage du texte

In [97]:
# Téléchargement des 51 fichiers PDF à partir des liens stockés dans le DataFrame
DOWNLOARD_DIR = "../data/tp1/pdfs"
os.makedirs(DOWNLOARD_DIR, exist_ok=True)

for index, row in df.iterrows():
    url = row["link"]
    response = requests.get(url, headers=headers)

    file_path = os.path.join(
        DOWNLOARD_DIR,
        row["pdf_name"]
    )

    with open(file_path, "wb") as writer:
        writer.write(response.content)

    print(f"{index + 1}/{len(df)} : {row['pdf_name']}")
    time.sleep(0.1)

1/51 : KB_JB230_1892-08-07_01-0003.pdf
2/51 : KB_JB427_1920-01-10_01-00004.pdf
3/51 : KB_JB555_1836-02-08_01-00002.pdf
4/51 : KB_JB638_1860-05-21_01-00002.pdf
5/51 : KB_JB773_1918-11-30_01-00002.pdf
6/51 : KB_JB838_1887-12-28_01-00003.pdf
7/51 : KB_JB230_1903-10-16_01-0002.pdf
8/51 : KB_JB427_1933-01-04_01-00003.pdf
9/51 : KB_JB555_1899-01-19_01-00003.pdf
10/51 : KB_JB638_1902-12-20_01-00002.pdf
11/51 : KB_JB773_1933-10-07_01-00007.pdf
12/51 : KB_JB838_1911-08-03_01-00006.pdf
13/51 : KB_JB230_1913-07-05_01-0001.pdf
14/51 : KB_JB427_1949-07-18_01-00008.pdf
15/51 : KB_JB555_1940-03-01_01-00004.pdf
16/51 : KB_JB638_1946-07-18_01-00003.pdf
17/51 : KB_JB773_1950-07-22_01-00010.pdf
18/51 : KB_JB838_1943-09-04_01-00002.pdf
19/51 : KB_JB258_1884-09-03_01-0003.pdf
20/51 : KB_JB449_1846-05-30_01-00002.pdf
21/51 : KB_JB567_1857-02-02_01-00003.pdf
22/51 : KB_JB685_1894-05-14_01-0003.pdf
23/51 : KB_JB835_1911-04-24_01-00004.pdf
24/51 : KB_JB92_1860-02-09_01-00003.pdf
25/51 : KB_JB258_1894-12-09_01-

In [98]:
## Confirmation de la présence des fichiers téléchargés
downloaded_pdfs = [
    filename
    for filename in os.listdir(DOWNLOARD_DIR)
    if filename.endswith(".pdf")
]

print(f"Nombre de PDF téléchargés : {len(downloaded_pdfs)}")

Nombre de PDF téléchargés : 51
